# 06. Build `df_patient` for SICI Sensitivity

This notebook prepares `df_patient` from the previous notebooks in this series.

Target columns:
- `sample`
- `group` (`R`/`NR`)
- `C_b_cyto`
- `C_malig_caf`
- `C_exh_cyto`

This notebook **stops after creating and exporting `df_patient`**.


## 1) Path bootstrap


In [10]:
import os
from pathlib import Path

# Resolve project root robustly (works from notebooks/, project root, or Downloads/).
_candidates = [
    Path.cwd(),
    Path.cwd().parent,
    Path.cwd() / "Spatial HCC",
    Path.cwd().parent / "Spatial HCC",
]
for _root in _candidates:
    if (_root / "GSE238264_RAW").exists() or (_root / "notebooks").exists():
        os.chdir(_root)
        break

print("Working directory:", Path.cwd())


Working directory: /Users/prateek/Downloads/Spatial HCC


## 2) Imports and helpers


In [11]:
import numpy as np
import pandas as pd


def normalize_group_label(x):
    """Normalize response labels into canonical R / NR strings."""
    s = str(x).strip().upper()
    if s in {"R", "RESPONDER"}:
        return "R"
    if s in {"NR", "NONRESPONDER", "NON-RESPONDER", "NON_RESPONDER"}:
        return "NR"
    return s


def check_required_columns(df, cols, name):
    """Raise helpful error if required columns are missing."""
    missing = [c for c in cols if c not in df.columns]
    if missing:
        raise KeyError(f"{name} missing required columns: {missing}")


## 3) Load patient-level couplings from previous notebooks


In [12]:
required_cols = ["sample", "group", "C_b_cyto", "C_malig_caf", "C_exh_cyto"]
metrics_source = None
source_name = None

# Preferred source: in-memory metrics_df from notebook 03.
if "metrics_df" in globals() and isinstance(metrics_df, pd.DataFrame):
    try:
        check_required_columns(metrics_df, required_cols, "metrics_df")
        metrics_source = metrics_df.copy()
        source_name = "in-memory metrics_df"
    except Exception:
        pass

# Fallback source: exported CSVs from notebook 03.
if metrics_source is None:
    candidate_paths = [
        Path("supplement_exports/Table_S1_metrics_couplings_torus.csv"),
        Path("supplement_exports_split/Table_S1_master_metrics_baselines.csv"),
        Path("supplement_exports_v2/Table_S1_master_metrics_baselines_torus.csv"),
    ]
    for p in candidate_paths:
        if p.exists():
            tmp = pd.read_csv(p)
            if all(c in tmp.columns for c in required_cols):
                metrics_source = tmp.copy()
                source_name = str(p)
                break

print("Selected source:", source_name)
if metrics_source is None:
    print("No metrics table found from previous notebooks.")


Selected source: supplement_exports/Table_S1_metrics_couplings_torus.csv


## 4) Last fallback: compute required couplings from `st_adata_scored.h5ad`


In [13]:
if metrics_source is None:
    import scanpy as sc
    import squidpy as sq
    from scipy.stats import spearmanr

    h5_path = Path("st_adata_scored.h5ad")
    if not h5_path.exists():
        raise FileNotFoundError(
            "Could not build df_patient. Missing both exported metrics tables and st_adata_scored.h5ad. "
            "Run notebook 02 and/or 03 first."
        )

    st_adata = sc.read_h5ad(h5_path.as_posix())
    req_obs = ["sample", "spot_exhaustion", "spot_cytotoxic", "spot_tam", "spot_caf", "spot_malignant", "spot_bcell"]
    check_required_columns(st_adata.obs, req_obs, "st_adata.obs")

    def edge_gradient_coupling(ad, key_a, key_b):
        """Compute coupling between two fields from edge gradients on grid neighbors."""
        from scipy.sparse import triu

        sq.gr.spatial_neighbors(ad, coord_type="grid")
        A = ad.obsp["spatial_connectivities"]
        U = triu(A, k=1).tocoo()
        i, j = U.row, U.col

        a = ad.obs[key_a].to_numpy()
        b = ad.obs[key_b].to_numpy()
        da = a[i] - a[j]
        db = b[i] - b[j]

        valid = np.isfinite(da) & np.isfinite(db)
        if valid.sum() < 10:
            return np.nan
        return float(spearmanr(da[valid], db[valid], nan_policy="omit")[0])

    st_adata.obs["sample"] = st_adata.obs["sample"].astype(str)
    st_by_sample = {
        s: st_adata[st_adata.obs["sample"] == s].copy()
        for s in sorted(st_adata.obs["sample"].unique())
    }

    # Build group labels from response if available, else infer from sample suffix.
    sample_groups = {}
    if "response" in st_adata.obs.columns:
        tmp = st_adata.obs[["sample", "response"]].dropna().copy()
        first_resp = tmp.groupby("sample", as_index=True)["response"].first()
        for s, r in first_resp.items():
            sample_groups[s] = normalize_group_label(r)
    for s in st_by_sample:
        if s not in sample_groups:
            su = s.upper()
            sample_groups[s] = "NR" if su.endswith("NR") else ("R" if su.endswith("R") else None)

    rows = []
    for s, ad in st_by_sample.items():
        rows.append({
            "sample": s,
            "group": sample_groups.get(s),
            "C_b_cyto": edge_gradient_coupling(ad, "spot_bcell", "spot_cytotoxic"),
            "C_malig_caf": edge_gradient_coupling(ad, "spot_malignant", "spot_caf"),
            "C_exh_cyto": edge_gradient_coupling(ad, "spot_exhaustion", "spot_cytotoxic"),
        })

    metrics_source = pd.DataFrame(rows)
    source_name = "computed from st_adata_scored.h5ad"
    print("Selected source:", source_name)


## 5) Build final `df_patient`


In [14]:
check_required_columns(metrics_source, required_cols, "metrics_source")

df_patient = metrics_source[required_cols].copy()
df_patient["sample"] = df_patient["sample"].astype(str)
df_patient["group"] = df_patient["group"].map(normalize_group_label)

# Keep only canonical groups and complete rows.
df_patient = df_patient[df_patient["group"].isin(["R", "NR"])].copy()
df_patient = df_patient.dropna(subset=["C_b_cyto", "C_malig_caf", "C_exh_cyto"])

# If duplicated samples exist (for example multiple scales), aggregate to one row per sample.
if df_patient["sample"].duplicated().any():
    df_patient = (
        df_patient.groupby("sample", as_index=False)
        .agg({
            "group": "first",
            "C_b_cyto": "mean",
            "C_malig_caf": "mean",
            "C_exh_cyto": "mean",
        })
    )

# Final sanity checks.
if set(df_patient["group"].unique()) != {"R", "NR"}:
    raise ValueError("df_patient must contain both R and NR groups.")

print("Source used:", source_name)
print("df_patient shape:", df_patient.shape)
print("Group counts:")
print(df_patient["group"].value_counts())
display(df_patient.sort_values("sample"))


Source used: supplement_exports/Table_S1_metrics_couplings_torus.csv
df_patient shape: (7, 5)
Group counts:
group
R     4
NR    3
Name: count, dtype: int64


,sample,group,C_b_cyto,C_malig_caf,C_exh_cyto
0,HCC1R,R,0.017711,-0.033758,0.019049
1,HCC2R,R,-0.032210,-0.057423,0.038262
2,HCC3R,R,0.007831,-0.040782,0.022126
3,HCC4R,R,0.095418,-0.035658,0.061939
4,HCC5NR,NR,-0.010910,-0.065038,-0.025713
5,HCC6NR,NR,0.002226,-0.104665,0.032672
6,HCC7NR,NR,0.032413,-0.070682,0.010254


## 6) Export `df_patient` for next analysis steps


In [15]:
out_dir = Path("sensitivity_outputs")
out_dir.mkdir(parents=True, exist_ok=True)

df_patient.to_csv(out_dir / "df_patient_for_sici_sensitivity.csv", index=False)
df_patient.to_pickle(out_dir / "df_patient_for_sici_sensitivity.pkl")

print("Saved:")
print("-", out_dir / "df_patient_for_sici_sensitivity.csv")
print("-", out_dir / "df_patient_for_sici_sensitivity.pkl")


Saved:
- sensitivity_outputs/df_patient_for_sici_sensitivity.csv
- sensitivity_outputs/df_patient_for_sici_sensitivity.pkl


In [16]:
import numpy as np
import pandas as pd

alpha = 1.0
beta = 1.0
gamma = 1.0 

df_patient["SICI_explicit"] = (
    alpha * df_patient["C_b_cyto"] +
    beta * df_patient["C_malig_caf"] +
    gamma * df_patient["C_exh_cyto"]        
)


In [17]:
from itertools import product

alphas = [0.5, 1.0, 2.0]
betas = [0.5, 1.0, 2.0]
gammas = [0.5, 1.0, 2.0]

results = []
for a, b, g in product(alphas, betas, gammas):
    sici = (
        a * df_patient["C_b_cyto"] +
        b * df_patient["C_malig_caf"] +
        g * df_patient["C_exh_cyto"]
    )

    mean_R = sici[df_patient["group"] == "R"].mean()
    mean_NR = sici[df_patient["group"] == "NR"].mean()
    diff = mean_R - mean_NR

    results.append({
        "alpha": a,
        "beta": b,
        "gamma": g,
        "mean_R": mean_R,
        "mean_NR": mean_NR,
        "diff_R_minus_NR": diff
    })

df_sici_grid = pd.DataFrame(results)
df_sici_grid = df_sici_grid.sort_values("diff_R_minus_NR", ascending=False).reset_index(drop=True)


In [22]:
df_sici_grid

,alpha,beta,gamma,mean_R,mean_NR,diff_R_minus_NR
0,2.0,2.0,2.0,0.031252,-0.132961,0.164214
1,1.0,2.0,2.0,0.009065,-0.140871,0.149936
2,0.5,2.0,2.0,-0.002029,-0.144826,0.142797
3,2.0,2.0,1.0,-0.004091,-0.138699,0.134608
4,2.0,1.0,2.0,0.073158,-0.052833,0.125991
5,1.0,2.0,1.0,-0.026279,-0.146609,0.120330
6,2.0,2.0,0.5,-0.021763,-0.141568,0.119805
7,0.5,2.0,1.0,-0.037373,-0.150564,0.113191
8,1.0,1.0,2.0,0.050970,-0.060743,0.111713
9,2.0,0.5,2.0,0.094110,-0.012769,0.106879


In [19]:
df_patient["SICI_no_Bcyto"] = (
beta * df_patient["C_malig_caf"] +
gamma * df_patient["C_exh_cyto"]
)

df_patient["SICI_no_MaligCAF"] = (
alpha * df_patient["C_b_cyto"] +
gamma * df_patient["C_exh_cyto"]
)               

df_patient["SICI_no_ExhCyto"] = (
alpha * df_patient["C_b_cyto"] +
beta * df_patient["C_malig_caf"]
)   


In [20]:
for col in ["SICI_explicit", "SICI_no_Bcyto", "SICI_no_MaligCAF", "SICI_no_ExhCyto"]:
    mean_R = df_patient.loc[df_patient["group"] == "R", col].mean()
    mean_NR = df_patient.loc[df_patient["group"] == "NR", col].mean()
    diff = mean_R - mean_NR
    print(f"{col}: mean R = {mean_R:.4f}, mean NR = {mean_NR:.4f}, diff = {diff:.4f}")

SICI_explicit: mean R = 0.0156, mean NR = -0.0665, diff = 0.0821
SICI_no_Bcyto: mean R = -0.0066, mean NR = -0.0744, diff = 0.0678
SICI_no_MaligCAF: mean R = 0.0575, mean NR = 0.0136, diff = 0.0439
SICI_no_ExhCyto: mean R = -0.0197, mean NR = -0.0722, diff = 0.0525


In [21]:
df_patient[["sample","SICI_explicit"]].sort_values("SICI_explicit")

,sample,SICI_explicit
4,HCC5NR,-0.101660
5,HCC6NR,-0.069766
1,HCC2R,-0.051372
6,HCC7NR,-0.028015
2,HCC3R,-0.010825
0,HCC1R,0.003003
3,HCC4R,0.121699


In [25]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import spearmanr

OUTDIR = "SICI_sensitivity_outputs"
os.makedirs(OUTDIR, exist_ok=True)

# -------------------------
# 0) Sanity checks
# -------------------------
required_cols = {"sample","group","C_b_cyto","C_malig_caf","C_exh_cyto"}
missing = required_cols - set(df_patient.columns)
if missing:
    raise KeyError(f"df_patient missing columns: {missing}")

if "diff_R_minus_NR" not in df_sici_grid.columns:
    raise KeyError("df_grid must have diff_R_minus_NR column.")

# Standardize group labels (just in case)
df_patient = df_patient.copy()
df_patient["group"] = df_patient["group"].astype(str)

# -------------------------
# 1) Robustness summary (grid)
# -------------------------
grid_summary = {
    "n_grid": int(df_sici_grid.shape[0]),
    "n_positive_diff": int((df_sici_grid["diff_R_minus_NR"] > 0).sum()),
    "frac_positive_diff": float((df_sici_grid["diff_R_minus_NR"] > 0).mean()),
    "diff_min": float(df_sici_grid["diff_R_minus_NR"].min()),
    "diff_median": float(df_sici_grid["diff_R_minus_NR"].median()),
    "diff_max": float(df_sici_grid["diff_R_minus_NR"].max()),
}
df_grid_summary = pd.DataFrame([grid_summary])
display(df_grid_summary)

df_sici_grid.to_csv(os.path.join(OUTDIR, "Table_SICI_coeff_grid_diff.csv"), index=False)
df_grid_summary.to_csv(os.path.join(OUTDIR, "Table_SICI_coeff_grid_summary.csv"), index=False)

# -------------------------
# 2) Rank stability across coefficient grid
#    (compare sample SICI ranks to default coefficients (1,1,1))
# -------------------------
def compute_sici(df, a, b, g, flip_exhcyto=False):
    s = (
        a * df["C_b_cyto"].values
        + b * df["C_malig_caf"].values
        + ( (-g) if flip_exhcyto else g ) * df["C_exh_cyto"].values
    )
    return pd.Series(s, index=df["sample"].values)

# Default coefficients
sici_default = compute_sici(df_patient, 1.0, 1.0, 1.0, flip_exhcyto=False)

stability_rows = []
for _, r in df_sici_grid.iterrows():
    a, b, g = float(r["alpha"]), float(r["beta"]), float(r["gamma"])
    sici_cur = compute_sici(df_patient, a, b, g, flip_exhcyto=False)

    # Spearman correlation of sample ranks (use values directly)
    rho, p = spearmanr(sici_default.values, sici_cur.values)
    stability_rows.append({
        "alpha": a, "beta": b, "gamma": g,
        "spearman_rho_vs_(1,1,1)": float(rho),
        "spearman_p": float(p),
        "diff_R_minus_NR": float(r["diff_R_minus_NR"]),
    })

df_stability = pd.DataFrame(stability_rows).sort_values(
    ["spearman_rho_vs_(1,1,1)", "diff_R_minus_NR"], ascending=[False, False]
)
display(df_stability.head(10))
df_stability.to_csv(os.path.join(OUTDIR, "Table_SICI_rank_stability_vs_default.csv"), index=False)

# -------------------------
# 3) Sensitivity plots (supp-ready)
# -------------------------
# 3a) Histogram of diff_R_minus_NR
plt.figure()
plt.hist(df_sici_grid["diff_R_minus_NR"].values, bins=10)
plt.xlabel("Δ = mean(SICI)_R − mean(SICI)_NR")
plt.ylabel("Count (coefficient settings)")
plt.title("SICI coefficient sensitivity (Δ across α,β,γ grid)")
plt.tight_layout()
plt.savefig(os.path.join(OUTDIR, "Fig_SICI_coeff_sensitivity_hist.png"), dpi=300)
plt.close()

# 3b) Heatmaps: fix gamma and plot alpha x beta for each gamma
for gamma_val in sorted(df_sici_grid["gamma"].unique()):
    sub = df_sici_grid[df_sici_grid["gamma"] == gamma_val].copy()
    pivot = sub.pivot(index="alpha", columns="beta", values="diff_R_minus_NR").sort_index().sort_index(axis=1)

    plt.figure()
    plt.imshow(pivot.values, aspect="auto")
    plt.xticks(range(pivot.shape[1]), pivot.columns.tolist())
    plt.yticks(range(pivot.shape[0]), pivot.index.tolist())
    plt.xlabel("beta (Malig–CAF weight)")
    plt.ylabel("alpha (B–Cyto weight)")
    plt.title(f"Δ = mean(SICI)_R − mean(SICI)_NR  (gamma={gamma_val})")
    plt.colorbar(label="Δ")
    plt.tight_layout()
    plt.savefig(os.path.join(OUTDIR, f"Fig_SICI_coeff_heatmap_gamma_{gamma_val}.png"), dpi=300)
    plt.close()

# -------------------------
# 4) Leave-one-term-out table (you already computed diffs; just format it)
# -------------------------
loo = pd.DataFrame([
    {"variant":"SICI_full (1,1,1)", "diff_R_minus_NR": 0.0821},
    {"variant":"SICI_no_Bcyto",     "diff_R_minus_NR": 0.0678},
    {"variant":"SICI_no_MaligCAF",  "diff_R_minus_NR": 0.0439},
    {"variant":"SICI_no_ExhCyto",   "diff_R_minus_NR": 0.0525},
]).sort_values("diff_R_minus_NR", ascending=False)

display(loo)
loo.to_csv(os.path.join(OUTDIR, "Table_SICI_leave_one_term_out.csv"), index=False)

# -------------------------
# 5) Flip-test for Exh–Cyto sign (optional but recommended)
# -------------------------
# This checks whether using -gamma*C_exh_cyto changes group separation direction/magnitude.
flip_rows = []
for _, r in df_sici_grid.iterrows():
    a, b, g = float(r["alpha"]), float(r["beta"]), float(r["gamma"])

    sici_plus  = compute_sici(df_patient, a, b, g, flip_exhcyto=False)
    sici_minus = compute_sici(df_patient, a, b, g, flip_exhcyto=True)

    tmp = df_patient[["sample","group"]].copy()
    tmp["sici_plus"]  = tmp["sample"].map(sici_plus)
    tmp["sici_minus"] = tmp["sample"].map(sici_minus)

    meanR_plus  = tmp.loc[tmp.group=="R",  "sici_plus"].mean()
    meanNR_plus = tmp.loc[tmp.group=="NR", "sici_plus"].mean()
    diff_plus   = meanR_plus - meanNR_plus

    meanR_minus  = tmp.loc[tmp.group=="R",  "sici_minus"].mean()
    meanNR_minus = tmp.loc[tmp.group=="NR", "sici_minus"].mean()
    diff_minus   = meanR_minus - meanNR_minus

    flip_rows.append({
        "alpha": a, "beta": b, "gamma": g,
        "diff_plus": float(diff_plus),
        "diff_minus": float(diff_minus),
        "diff_minus_minus_plus": float(diff_minus - diff_plus),
    })

df_flip = pd.DataFrame(flip_rows)
df_flip.to_csv(os.path.join(OUTDIR, "Table_SICI_exhcyto_flip_test.csv"), index=False)

# Plot: diff_plus vs diff_minus scatter
plt.figure()
plt.scatter(df_flip["diff_plus"].values, df_flip["diff_minus"].values)
plt.xlabel("Δ with +gamma*C_exh_cyto")
plt.ylabel("Δ with -gamma*C_exh_cyto")
plt.title("Flip-test: Exhaustion–Cytotoxic coupling sign sensitivity")
plt.tight_layout()
plt.savefig(os.path.join(OUTDIR, "Fig_SICI_exhcyto_flip_scatter.png"), dpi=300)
plt.close()

print("Saved supplement exports to:", OUTDIR)

,n_grid,n_positive_diff,frac_positive_diff,diff_min,diff_median,diff_max
0,27,27,1.0,0.041053,0.096385,0.164214


,alpha,beta,gamma,"spearman_rho_vs_(1,1,1)",spearman_p,diff_R_minus_NR
0,2.0,2.0,2.0,1.000000,0.000000,0.164214
3,2.0,2.0,1.0,1.000000,0.000000,0.134608
14,1.0,0.5,2.0,1.000000,0.000000,0.092601
16,1.0,1.0,1.0,1.000000,0.000000,0.082107
20,1.0,1.0,0.5,1.000000,0.000000,0.067304
26,0.5,0.5,0.5,1.000000,0.000000,0.041053
1,1.0,2.0,2.0,0.964286,0.000454,0.149936
2,0.5,2.0,2.0,0.964286,0.000454,0.142797
4,2.0,1.0,2.0,0.964286,0.000454,0.125991
5,1.0,2.0,1.0,0.964286,0.000454,0.120330


,variant,diff_R_minus_NR
0,"SICI_full (1,1,1)",0.0821
1,SICI_no_Bcyto,0.0678
3,SICI_no_ExhCyto,0.0525
2,SICI_no_MaligCAF,0.0439


Saved supplement exports to: SICI_sensitivity_outputs


In [27]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import spearmanr

OUTDIR = "supplement_exports"
os.makedirs(OUTDIR, exist_ok=True)

# Ensure group labels consistent
df_patient = df_patient.copy()
df_patient["group"] = df_patient["group"].astype(str)

# Function to compute SICI
def compute_sici(df, a, b, g, flip_exh=False):
    if flip_exh:
        return (
            a * df["C_b_cyto"].values
            + b * df["C_malig_caf"].values
            - g * df["C_exh_cyto"].values
        )
    else:
        return (
            a * df["C_b_cyto"].values
            + b * df["C_malig_caf"].values
            + g * df["C_exh_cyto"].values
        )

flip_results = []

for _, row in df_sici_grid.iterrows():
    a = float(row["alpha"])
    b = float(row["beta"])
    g = float(row["gamma"])

    sici_plus  = compute_sici(df_patient, a, b, g, flip_exh=False)
    sici_minus = compute_sici(df_patient, a, b, g, flip_exh=True)

    tmp = df_patient[["sample","group"]].copy()
    tmp["plus"]  = sici_plus
    tmp["minus"] = sici_minus

    meanR_plus  = tmp.loc[tmp.group=="R","plus"].mean()
    meanNR_plus = tmp.loc[tmp.group=="NR","plus"].mean()
    diff_plus   = meanR_plus - meanNR_plus

    meanR_minus  = tmp.loc[tmp.group=="R","minus"].mean()
    meanNR_minus = tmp.loc[tmp.group=="NR","minus"].mean()
    diff_minus   = meanR_minus - meanNR_minus

    rho, _ = spearmanr(tmp["plus"], tmp["minus"])

    flip_results.append({
        "alpha": a,
        "beta": b,
        "gamma": g,
        "diff_plus": diff_plus,
        "diff_minus": diff_minus,
        "diff_difference": diff_minus - diff_plus,
        "rank_rho_plus_vs_minus": rho
    })

df_flip_summary = pd.DataFrame(flip_results)

# Summary statistics
summary = {
    "n_settings": len(df_flip_summary),
    "n_plus_positive": int((df_flip_summary["diff_plus"] > 0).sum()),
    "n_minus_positive": int((df_flip_summary["diff_minus"] > 0).sum()),
    "frac_minus_positive": float((df_flip_summary["diff_minus"] > 0).mean()),
    "diff_plus_min": float(df_flip_summary["diff_plus"].min()),
    "diff_plus_max": float(df_flip_summary["diff_plus"].max()),
    "diff_minus_min": float(df_flip_summary["diff_minus"].min()),
    "diff_minus_max": float(df_flip_summary["diff_minus"].max()),
    "median_rank_rho": float(df_flip_summary["rank_rho_plus_vs_minus"].median())
}

df_flip_summary_stats = pd.DataFrame([summary])

display(df_flip_summary_stats)
display(df_flip_summary.head())

# Save tables
df_flip_summary.to_csv(os.path.join(OUTDIR, "Table_SICI_flip_test_full.csv"), index=False)
df_flip_summary_stats.to_csv(os.path.join(OUTDIR, "Table_SICI_flip_test_summary.csv"), index=False)

# Scatter plot
plt.figure()
plt.scatter(df_flip_summary["diff_plus"], df_flip_summary["diff_minus"])
plt.axhline(0, linestyle="--")
plt.axvline(0, linestyle="--")
plt.xlabel("Δ (Exh–Cyto positive)")
plt.ylabel("Δ (Exh–Cyto flipped negative)")
plt.title("Flip Test: Effect of Exhaustion–Cytotoxic Sign on SICI Separation")
plt.tight_layout()
plt.savefig(os.path.join(OUTDIR, "Fig_SICI_flip_test_scatter.png"), dpi=300)
plt.close()

print("Flip test results saved to:", OUTDIR)

,n_settings,n_plus_positive,n_minus_positive,frac_minus_positive,diff_plus_min,diff_plus_max,diff_minus_min,diff_minus_max,median_rank_rho
0,27,27,21,0.777778,0.041053,0.164214,-0.032961,0.090199,0.714286


,alpha,beta,gamma,diff_plus,diff_minus,diff_difference,rank_rho_plus_vs_minus
0,2.0,2.0,2.0,0.164214,0.045791,-0.118423,0.714286
1,1.0,2.0,2.0,0.149936,0.031513,-0.118423,0.357143
2,0.5,2.0,2.0,0.142797,0.024374,-0.118423,-0.035714
3,2.0,2.0,1.0,0.134608,0.075396,-0.059212,0.857143
4,2.0,1.0,2.0,0.125991,0.007567,-0.118423,0.464286


Flip test results saved to: supplement_exports


In [28]:
bad = df_flip_summary[df_flip_summary["diff_minus"] < 0].sort_values("diff_minus")
display(bad[["alpha","beta","gamma","diff_plus","diff_minus","rank_rho_plus_vs_minus"]])
print("Bad settings count:", bad.shape[0])

,alpha,beta,gamma,diff_plus,diff_minus,rank_rho_plus_vs_minus
15,0.5,0.5,2.0,0.085462,-0.032961,-0.678571
14,1.0,0.5,2.0,0.092601,-0.025822,-0.035714
11,0.5,1.0,2.0,0.104574,-0.013850,-0.107143
9,2.0,0.5,2.0,0.106879,-0.011544,0.571429
8,1.0,1.0,2.0,0.111713,-0.006711,-0.035714
24,0.5,0.5,1.0,0.055856,-0.003355,-0.035714


Bad settings count: 6
